# Post-Training Steps 2 & 4: RLVR with GRPO

**Paper §3.2** — Multi-environment Reinforcement Learning from Verifiable Rewards
using synchronous GRPO.

RLVR is applied **twice** in the full pipeline:
- **Step 2** — immediately after SFT (this notebook)
- **Step 4** — after RLHF (re-run this notebook loading from the RLHF checkpoint)

### Key design choices
| Feature | Detail |
|---|---|
| Algorithm | Synchronous GRPO |
| Reward | Verifiable (format + correctness) |
| Curriculum | Difficulty increases easy → hard (§3.2.2) |
| Router freezing | MoE router weights frozen during RL (§3.2.5) |
| Overlong filter | Completions at the length cap are discarded |
| MoE biases | Aux-loss-free bias update kept active |

### Notebook contents
1. Environment setup
2. Imports & hyperparameters
3. Tokenizer
4. Dataset
5. Model (load SFT checkpoint)
6. RLVR helpers
7. Training loop


## 0. Environment Setup

Detects Colab / Kaggle, installs packages, and adds the repo to `sys.path`.

In [ ]:
import os, sys

IN_COLAB  = False
IN_KAGGLE = os.path.exists("/kaggle/working") and os.path.exists("/kaggle/input")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print(f"Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE}")


In [ ]:
if IN_COLAB or IN_KAGGLE:
    # Set ACCELERATOR to "cuda12" (GPU) or "tpu" (Colab TPU only).
    ACCELERATOR = "cuda12"
    import subprocess
    subprocess.run(
        ["pip", "install", "-q",
         f"jax[{ACCELERATOR}]", "flax", "optax", "orbax-checkpoint",
         "datasets", "transformers", "matplotlib"],
        check=True,
    )


In [ ]:
import pathlib

REPO_URL = "https://github.com/wisnunugroho21/nugie-jax-nemotron.git"

if IN_COLAB:
    REPO_DIR = pathlib.Path("/content/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    sys.path.insert(0, str(REPO_DIR))
elif IN_KAGGLE:
    REPO_DIR = pathlib.Path("/kaggle/working/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    sys.path.insert(0, str(REPO_DIR))
else:
    # Local: assume the notebook lives inside the repo
    REPO_DIR = pathlib.Path(".").resolve().parent
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))


## 1. Imports & Hyperparameters

In [ ]:
import pathlib

import jax
import jax.numpy as jnp
import numpy as np
from flax import nnx
from transformers import AutoTokenizer

from moe import SparseMoE
from nemotron import NemotronConfig, NemotronNanoBlock
from training_shared import (
    SFT_CKPT_DIR, RLVR_CKPT_DIR, RL_TRAIN_SEQ_LEN,
    RLVR_NUM_PROMPTS, RLVR_NUM_GENERATIONS, RLVR_STEPS,
    RLVR_MAX_NEW_TOKENS, RLVR_FREEZE_ROUTER,
    RLVR_LR, RLVR_WD, RLVR_B1, RLVR_B2, RLVR_CKPT_EVERY,
    build_model, collect_moe_layers, make_constant_lr_optimizer,
    load_sft_data, generate_completion_tokens,
    compute_verifiable_reward, compute_grpo_advantages,
    curriculum_sample_indices, build_grpo_batch,
    rl_step, compute_log_probs, update_moe_biases,
    make_checkpoint_manager, save_checkpoint, try_load_from_dir,
)

print("JAX devices:", jax.devices())


In [ ]:
TRAIN_STEPS   = RLVR_STEPS            # 100
NUM_PROMPTS   = RLVR_NUM_PROMPTS      # 4
NUM_GENS      = RLVR_NUM_GENERATIONS  # 4
FREEZE_ROUTER = RLVR_FREEZE_ROUTER    # True
CKPT_DIR      = RLVR_CKPT_DIR

print(f"RLVR: STEPS={TRAIN_STEPS} | prompts/step={NUM_PROMPTS} | gens/prompt={NUM_GENS}")
print(f"Freeze MoE router: {FREEZE_ROUTER}")


## 2. Tokenizer

In [ ]:
print("Loading Nemotron tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained("nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size : {tokenizer.vocab_size}")


## 3. Dataset

GSM8K provides verifiable ground-truth answers (numeric) used for the reward signal.

In [ ]:
train_samples = load_sft_data("train")
print(f"Train samples : {len(train_samples)}")
print(f"Example: {train_samples[0]['question'][:80]}...")


## 4. Model — Load SFT Checkpoint

In [ ]:
print("Building model ...")
model = build_model(seed=0)
moe_layers = collect_moe_layers(model)

loaded = try_load_from_dir(SFT_CKPT_DIR, model, model.config)
if loaded:
    print("Resumed from SFT checkpoint.")
else:
    print("No SFT checkpoint found — starting from random init.")

# Frozen reference policy for KL penalty.
graphdef, ref_state = nnx.split(model)
ref_model = nnx.merge(graphdef, ref_state)


## 5. Optimizer

In [ ]:
tx        = make_constant_lr_optimizer(RLVR_LR, RLVR_WD, RLVR_B1, RLVR_B2)
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)
ckpt_mgr  = make_checkpoint_manager(CKPT_DIR)
print(f"Constant-LR AdamW (lr={RLVR_LR}).")


## 6. Router Freezing Helpers

**Paper §3.2.5** — freezing router weights during RL stabilises training by preventing
the routing distribution from shifting while policy gradients update the expert weights.

In [ ]:
def snapshot_router_kernels(moe_layers: list[SparseMoE]) -> list:
    """Copy router kernels so they can be restored after an optimizer step."""
    return [moe.router.kernel.get_value() for moe in moe_layers]

def restore_router_kernels(moe_layers: list[SparseMoE], kernels: list) -> None:
    """Restore router kernels to enforce router freezing during RLVR."""
    for moe, kernel in zip(moe_layers, kernels):
        moe.router.kernel.set_value(kernel)


## 7. Training Loop

At each step:
1. Curriculum-sample a mini-batch of prompts
2. Generate `NUM_GENS` completions per prompt
3. Compute verifiable rewards → GRPO advantages
4. Build padded token batch and run a policy-gradient update
5. Restore router kernels (if frozen)
6. Update MoE biases

In [ ]:
print(f"\n=== Post-Training Step 2: RLVR (GRPO) ===")
print(f"Training for {TRAIN_STEPS} steps ...\n")

# Initial pass-rates for curriculum sampling (0.5 = medium difficulty).
pass_rates = np.full(len(train_samples), 0.5, dtype=np.float32)
reward_history: list[float] = []

for step in range(TRAIN_STEPS):
    batch_indices = curriculum_sample_indices(
        pass_rates=pass_rates, step=step, total_steps=TRAIN_STEPS,
        batch_size=NUM_PROMPTS,
    )
    batch_samples = [train_samples[i] for i in batch_indices]

    prompt_ids_list:   list[list[int]]       = []
    completion_groups: list[list[list[int]]] = []
    reward_groups:     list[list[float]]     = []

    for sample in batch_samples:
        prompt_text  = f"User: {sample['question']}\nAssistant: "
        p_ids        = tokenizer.encode(prompt_text, add_special_tokens=False)
        ground_truth = sample["answer"].split("####")[-1].strip()
        prompt_ids_list.append(p_ids)

        completions: list[list[int]] = []
        rewards:     list[float]     = []
        for g in range(NUM_GENS):
            comp_ids = generate_completion_tokens(
                model, tokenizer, p_ids, rng_seed=step * 100 + g
            )
            if len(comp_ids) >= RLVR_MAX_NEW_TOKENS:
                comp_ids = []; rewards.append(0.0)
            else:
                comp_text = tokenizer.decode(comp_ids, skip_special_tokens=True)
                rewards.append(compute_verifiable_reward(comp_text, ground_truth))
            completions.append(comp_ids)

        completion_groups.append(completions)
        reward_groups.append(rewards)

    advantage_groups = [compute_grpo_advantages(rg) for rg in reward_groups]
    token_ids, masks, advantages = build_grpo_batch(
        prompt_ids_list, completion_groups, advantage_groups,
        seq_len=RL_TRAIN_SEQ_LEN, pad_id=tokenizer.eos_token_id,
    )
    ref_log_probs = compute_log_probs(ref_model, token_ids)
    old_log_probs = compute_log_probs(model, token_ids)

    router_kernels = snapshot_router_kernels(moe_layers) if FREEZE_ROUTER else None
    loss = rl_step(model, optimizer, token_ids, masks, advantages, ref_log_probs, old_log_probs)
    if router_kernels is not None:
        restore_router_kernels(moe_layers, router_kernels)
    update_moe_biases(moe_layers)

    # Update per-sample pass-rates for next curriculum step.
    pass_rates[batch_indices] = np.array(
        [float(np.mean(np.array(rewards) > 0.0)) for rewards in reward_groups],
        dtype=np.float32,
    )

    mean_reward = float(np.mean([r for rg in reward_groups for r in rg]))
    reward_history.append(mean_reward)

    if step % 10 == 0:
        print(f"  RLVR step {step:3d} | loss={float(loss):.4f} | mean_reward={mean_reward:.3f}")

    if step % RLVR_CKPT_EVERY == 0:
        save_checkpoint(ckpt_mgr, model, step)

save_checkpoint(ckpt_mgr, model, TRAIN_STEPS)
print("\nRLVR complete.")


## 8. Reward Curve

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, len(reward_history) + 1), reward_history)
    plt.xlabel("Step"); plt.ylabel("Mean Reward"); plt.title("RLVR Mean Reward per Step")
    plt.grid(True); plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot.")
